# Model Collapse & Induction Heads — run notebook

Skeleton driver. All logic lives in the modules (`transformer.py`, `data.py`,
`collapse.py`, `analysis.py`); this notebook only wires them together and plots.
Fill in the stubs in those modules first — cells below will error until you do.

Workflow: train (checkpoints each generation to disk) → load checkpoints → analyse.

In [ ]:
# Install dependencies into this kernel (run once).
%pip install -r requirements.txt

In [ ]:
%load_ext autoreload
%autoreload 2

import jax, jax.numpy as jnp, jax.random as jr
import numpy as np
import matplotlib.pyplot as plt

import collapse as C
import data as D
import analysis as A

## Config

In [ ]:
cfg = C.Config(
    feature_dim=64, n_train_classes=50, n_val_classes=20, n_test_classes=30,
    n_labels=5, context_len=2, burstiness=1,
    d_model=64, depth=2, num_heads=8,
    batch_size=32, lr=1e-5, train_iters=50_000, eval_every=2_000,
    mode='base', n_generations=8, run_name='base_run',
)
cfg

## Generation 0 — does the task learn?

Build data + model, train one generation, and confirm it reaches high query
accuracy before committing to the full collapse loop.

In [ ]:
key = jr.PRNGKey(cfg.init_seed)
n_classes = cfg.n_train_classes + cfg.n_val_classes + cfg.n_test_classes
emb = D.make_embeddings(key, n_classes, cfg.n_exemplars, cfg.feature_dim)
train_cls, val_cls, test_cls = D.split_classes(cfg)

train_sampler = D.build_sampler(cfg, emb, train_cls)
val_sampler   = D.build_sampler(cfg, emb, val_cls)

model0 = C.build_model(cfg, key)
model0, hist0 = C.train_generation(cfg, model0, train_sampler, val_sampler, key)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(hist0['step'], hist0['train_loss'], label='train')
ax[0].plot(hist0['step'], hist0['val_loss'], label='val')
ax[0].set(xlabel='step', ylabel='loss'); ax[0].legend()
ax[1].plot(hist0['step'], hist0['val_acc'])
ax[1].set(xlabel='step', ylabel='val accuracy'); plt.tight_layout()

## Collapse pipeline
Trains gen 0..N, checkpointing each generation under `checkpoints/{run_name}/`.

In [ ]:
models, histories = C.collapse_pipeline(cfg)

## Metrics across generations
Reload from checkpoints so this works in a fresh kernel too.

In [ ]:
gens = range(cfg.n_generations + 1)
models = [C.load_checkpoint(cfg, f'checkpoints/{cfg.run_name}/gen{g}/model.eqx',
                            kind=cfg.mode) for g in gens]

acc  = [C.accuracy(m, val_sampler, jr.fold_in(key, g), n_batches=20)
        for g, m in zip(gens, models)]
ent  = [A.label_entropy(m, val_sampler, jr.fold_in(key, 100 + g), n_batches=20)
        for g, m in zip(gens, models)]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(list(gens), acc, 'o-'); ax[0].set(xlabel='generation', ylabel='val accuracy')
ax[1].plot(list(gens), ent, 'o-'); ax[1].set(xlabel='generation', ylabel='label entropy')
plt.tight_layout()

## Interpretability
Previous-token head (L0), induction head (L1), ablation, embeddings.

In [ ]:
batch = D.next_batch(val_sampler, jr.PRNGKey(7))

# Attention + circuit strength across generations
pt  = [A.prev_token_score(A.get_activations(m, batch), layer=0) for m in models]
ind = [A.induction_strength(A.get_activations(m, batch), batch['class_ids'], layer=1, head=0)
       for m in models]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(list(gens), pt, 'o-');  ax[0].set(xlabel='generation', ylabel='prev-token score')
ax[1].plot(list(gens), ind, 'o-'); ax[1].set(xlabel='generation', ylabel='induction strength')
plt.tight_layout()

In [ ]:
# One attention map (gen 0, layer 1, head 0)
amap = A.attention_maps(A.get_activations(models[0], batch), layer=1, head=0)[0]
plt.figure(figsize=(4, 4)); plt.imshow(amap); plt.colorbar()
plt.title('L1H0 attention — gen 0'); plt.xlabel('key pos'); plt.ylabel('query pos')

In [ ]:
# Head ablation sweep (which heads carry the circuit) and embedding geometry
drop = A.head_ablation_sweep(cfg, models[0], val_sampler, jr.PRNGKey(3))
W = A.symbol_embeddings(models[0])  # feed to PCA/t-SNE